# IMDb Sentiment Analysis
Performs inference using three different architectural approaches:

1. Scikit-Learn -> Native Pandas: Single-node driver inference (low latency for small datasets)
2. Scikit-Learn -> Spark UDF: Distributed cluster interence via Apache Arrow (scales single-node packages to billions of rows)
3. SparkML -> Native Spark Pipeline: High-performance JVM-native execution using Spark's `.transform()` engine


In [0]:
import mlflow
import pandas as pd

from pyspark.sql.functions import col, struct

In [0]:
SKLEARN_MODEL_URI = "models:/imdb_sklearn_model/1"
SPARKML_MODEL_URI = "models:/imdb_sparkml_model/1"

In [0]:
# 1. Load the model as a generic Python function
pandas_model = mlflow.pyfunc.load_model(SKLEARN_MODEL_URI)
#pandas_model = mlflow.sklearn.load_model(SKLEARN_MODEL_URI)

# 2. Load the test data into a Spark DataFrame and then convert to Pandas for scikit-learn compatibility
spark_test_df = spark.table("imdb_sentiment_test")
pandas_test_df = spark_test_df.toPandas()

# 3. Predict on the test data
X_pandas = pandas_test_df['text']
pandas_test_df['prediction'] = pandas_model.predict(X_pandas)

pandas_predictions_df = pandas_test_df[['text', 'label','prediction']]

# 4. Convert predictions to a Spark DataFrame and save to a Delta table
spark_predictions_df = spark.createDataFrame(pandas_predictions_df)

spark_predictions_df.write \
  .format("delta") \
  .mode("overwrite") \
  .saveAsTable("imdb_sentiment_predictions")

print("Successfully saved predictions to a Delta Table!")

In [0]:
%sql
select * from imdb_sentiment_predictions;

In [0]:
# 1. Load the model as a Spark UDF 
spark_predict_udf = mlflow.pyfunc.spark_udf(spark, model_uri=SKLEARN_MODEL_URI)

# 2. Load the data into a Spark DataFrame
spark_test_df = spark.table("imdb_sentiment_test")

# 3. Predict on the test data
spark_predictions_df = (spark_test_df
    #.withColumn("prediction", spark_predict_udf(struct(*[col(c) for c in ['text']]))[0])
    .withColumn("prediction", spark_predict_udf(col("text"))[0])
    .select("text", "label", "prediction")
)


# 4. Save predictions to a Delta table
spark_predictions_df.write \
  .format("delta") \
  .mode("overwrite") \
  .saveAsTable("imdb_predictions")

print("Successfully saved predictions to a Delta Table!")

In [0]:
%sql
select * from imdb_sentiment_predictions;